In [23]:
import pandas as pd
import numpy as np
import mlflow
from mlflow.sklearn import SERIALIZATION_FORMAT_CLOUDPICKLE
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import optuna

In [2]:
df = pd.read_csv("../data/processed/jakarta_properties_processed_tes.csv")

In [3]:
df.head()

,price_idr,district,bedrooms,bathrooms,garage,land_size_m2,building_size_m2,cluster,pool,mrt,tol,mall,city_Jakarta Barat,city_Jakarta Pusat,city_Jakarta Selatan,city_Jakarta Timur,city_Jakarta Utara,sub_district
0,21.311053,pesanggrahan,3.0,2.0,2.0,4.795791,4.394449,0,0,0,0,0,0,0,1,0,0,pesanggrahan
1,22.654787,tebet,4.0,4.0,4.0,4.343805,5.929589,0,0,0,0,0,0,0,1,0,0,tebet
2,23.025851,kembangan,4.0,4.0,2.0,5.420535,6.216606,0,0,0,0,0,1,0,0,0,0,puri indah
3,21.787977,kelapa gading,4.0,4.0,0.0,4.634729,4.634729,0,0,0,0,0,0,0,0,0,1,kelapa gading
4,21.947041,duren sawit,3.0,2.0,4.0,4.663439,4.691348,1,0,0,0,0,0,0,0,1,0,pondok kelapa


In [4]:
models_dict = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=20,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        objective='reg:absoluteerror',
        max_depth=10,
        learning_rate=0.05953139963770414,
        n_estimators=2747,
        subsample=0.8337433250519926,
        colsample_bytree=0.7543943678414979,
        gamma=0.032771478039126015,
        min_child_weight=6,
        reg_alpha=1.716855990003528,
        reg_lambda=6.819289477686187,
        random_state=42
    )
}

In [5]:
X = df.drop(columns=["price_idr"])
y = df["price_idr"]

# te_district = TargetEncoder(
#     smooth=1,
#     cv=5,
#     target_type='continuous'
#     )
#
# te_subdistrict = TargetEncoder(
#     smooth=1,
#     cv=5,
#     target_type='continuous'
#     )



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# X_train['district'] = te_district.fit_transform(X_train[['district']], y_train).flatten()
# X_test['district'] = te_district.transform(X_test[['district']]).flatten()
#
# X_train['sub_district'] = te_subdistrict.fit_transform(X_train[['sub_district']], y_train).flatten()
# X_test['sub_district'] = te_subdistrict.transform(X_test[['sub_district']]).flatten()

In [ ]:
cat_cols = X_train.select_dtypes(exclude=np.number).columns
num_cols = X_train.select_dtypes(include=np.number).columns
encoder = ColumnTransformer([
    (
        "target_encoder",
        TargetEncoder(
            smooth=1,
            cv=5,
            target_type="continuous"
        ),
        cat_cols
    ),
    ("numerical", "passthrough", num_cols),
],verbose_feature_names_out=False)

In [26]:
mlflow.set_tracking_uri("http://localhost:5000/")

In [31]:
def cross_validation(model, X_train, y_train, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_binned = pd.qcut(y_train, q=10, labels=False)

    mlflow.set_experiment("Housing Price Jakarta - Cross Validation")
    for model_name, model in models_dict.items():
        pipeline = Pipeline([
        ("encoder",encoder),
        ("model",model)
        ])
        print(f"\nCross Validation {model_name} ...")
        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=skf.split(X_train, y_binned),
            scoring='r2'
        )
        model_fitted = pipeline.named_steps['model']
        print(f"R2 per fold : {scores}")
        print(f"Rata-rata R2: {scores.mean():.4f}")
        print(f"Std deviasi  : {scores.std():.4f}")
        with mlflow.start_run(run_name=f"{model_name}-CV"):
            mlflow.log_params(model.get_params())
            mlflow.log_metric("R2 Mean",scores.mean())
            mlflow.log_metric("STD",scores.std())

            mlflow.sklearn.log_model(
                sk_model=pipeline,
                name=f"{model_name.lower().replace(' ', '_')}_model",
                serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
            )

In [36]:
cross_validation(model=models_dict, X_train=X_train, y_train=y_train, n_splits=5)

2026/08/06 00:53:58 INFO mlflow.tracking.fluent: Experiment with name 'Housing Price Jakarta - Cross Validation' does not exist. Creating a new experiment.



Cross Validation Linear Regression ...


2026/08/06 00:53:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


R2 per fold : [0.87230069 0.87428901 0.87485315 0.87597619 0.87188513]
Rata-rata R2: 0.8739
Std deviasi  : 0.0015
🏃 View run Linear Regression-CV at: http://localhost:5000/#/experiments/1/runs/3c4a9c36a02242c888f08cff2bf5cb61
🧪 View experiment at: http://localhost:5000/#/experiments/1

Cross Validation Decision Tree ...


2026/08/06 00:54:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


R2 per fold : [0.87424539 0.87490326 0.87348701 0.87734584 0.8768442 ]
Rata-rata R2: 0.8754
Std deviasi  : 0.0015
🏃 View run Decision Tree-CV at: http://localhost:5000/#/experiments/1/runs/b0eaef80bb5746b88857156b00922250
🧪 View experiment at: http://localhost:5000/#/experiments/1

Cross Validation Random Forest ...
R2 per fold : [0.88523562 0.88375935 0.88353129 0.8855366  0.88257634]
Rata-rata R2: 0.8841
Std deviasi  : 0.0011


2026/08/06 00:54:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest-CV at: http://localhost:5000/#/experiments/1/runs/97853348f130479db78b5de0defb2d11
🧪 View experiment at: http://localhost:5000/#/experiments/1

Cross Validation XGBoost ...


2026/08/06 00:56:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


R2 per fold : [0.9192102  0.92028529 0.92250094 0.92322539 0.9211751 ]
Rata-rata R2: 0.9213
Std deviasi  : 0.0015
🏃 View run XGBoost-CV at: http://localhost:5000/#/experiments/1/runs/bdfe72508a904cfe93d706a874bab6d5
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [ ]:
import json


def train_single_model(model_name,model,X_train,y_train,X_test,y_test):
    pipeline = Pipeline([
    ("encoder", encoder),
    ("model", model)
    ])
    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_test_true = np.expm1(y_test)
    y_pred_true = np.expm1(y_pred)
    error_pct = (np.abs(y_pred_true - y_test_true) / y_test_true)
    mlflow.set_experiment("Housing Price Jakarta - Single Model")

    result = {
        "Model" : model_name,
        "R2 Score":r2_score(y_test,y_pred),
        "MAE (mean)":mean_absolute_error(y_test_true,y_pred_true),
        "MDAE (median)": median_absolute_error(y_test_true,y_pred_true),
        "MAPE": mean_absolute_percentage_error(y_test_true,y_pred_true),
        "Q25": error_pct.quantile(0.25),
        "Q50": error_pct.quantile(0.50),
        "Q75": error_pct.quantile(0.75)
    }
    with open('../artifacts/models/metrics_model.json', 'w') as f:
        json.dump(result, f, ensure_ascii=False, indent=4)

    # FEATURE IMPORTANCE
    fitted_preprocessor = pipeline.named_steps["encoder"]
    fitted_model = pipeline.named_steps["model"]

    with mlflow.start_run(run_name=f"single-{model_name}"):
        mlflow.set_tags({
            "stage": "final_training",
            "evaluation_type": "holdout_test_set",
            "model_name": model_name,
        })

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "test_r2": result["R2 Score"],
            "test_mae": result["MAE (mean)"],
            "test_mdae": result["MDAE (median)"],
            "test_mape": result["MAPE"],
            "test_q25": result["Q25"],
            "test_q50": result["Q50"],
            "test_q75": result["Q75"],})

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name=f"{model_name.lower().replace(' ', '_')}_final_model",
            serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE
        )


    feature_names = fitted_preprocessor.get_feature_names_out()

    feat_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": fitted_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print(feat_imp)


In [ ]:
cv =KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective Function
def objective(trial):

    params = {
        "objective": "reg:absoluteerror",
        "max_depth": trial.suggest_int("max_depth", 3, 10),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-3,
            0.3,
            log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            4000
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        "random_state": 42,
    }

    model = XGBRegressor(**params)

    score = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=200
)

print("Best Score :", study.best_value)
print("Best Params:", study.best_params)

best param = {'max_depth': 10, 'learning_rate': 0.05953139963770414, 'n_estimators': 2747, 'subsample': 0.8337433250519926, 'colsample_bytree': 0.7543943678414979, 'gamma': 0.032771478039126015, 'min_child_weight': 6, 'reg_alpha': 1.716855990003528, 'reg_lambda': 6.819289477686187}


In [37]:
train_single_model(
    model_name = "XGBoost",
    model = models_dict["XGBoost"],
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

2026/08/06 00:56:54 INFO mlflow.tracking.fluent: Experiment with name 'Housing Price Jakarta - Single Model' does not exist. Creating a new experiment.
2026/08/06 00:56:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run single-XGBoost at: http://localhost:5000/#/experiments/2/runs/897181fe93f04acb829baf1def7ed099
🧪 View experiment at: http://localhost:5000/#/experiments/2
                 feature  importance
5           land_size_m2    0.113013
15    city_Jakarta Timur    0.077617
6       building_size_m2    0.068283
16    city_Jakarta Utara    0.066325
1           sub_district    0.061524
14  city_Jakarta Selatan    0.056534
12    city_Jakarta Barat    0.055914
9                    mrt    0.054770
8                   pool    0.052735
0               district    0.052349
10                   tol    0.052195
11                  mall    0.051860
13    city_Jakarta Pusat    0.051480
7                cluster    0.048065
3              bathrooms    0.046862
4                 garage    0.046247
2               bedrooms    0.044226
